# Práctica — Clase 10: completa el `worker.py` real de RAG sobre PDF

Este ejercicio reproduce, casi línea por línea, la pareja de plantilla y
solución más clara de todo el material de IBM: el proyecto
`wbphl-build_own_chatbot_without_open_ai` ("Personal Data Assistant")
distribuye un `worker.py` con varios huecos sin resolver, y un
`Worker_completed.py` con la solución exacta de esos mismos huecos.

Vas a completar tú los 4 huecos reales:

1. Crear el modelo de embeddings.
2. Cargar el PDF con `PyPDFLoader`.
3. Trocear el texto con `RecursiveCharacterTextSplitter`.
4. Guardar cada pregunta y respuesta en `chat_history`.

**Nota sobre el LLM**: el `worker.py` original usa `WatsonxLLM` (de pago).
Aquí usamos la variante 100% gratuita que el propio repositorio de IBM
incluye en `worker_huggingFace.py`: `HuggingFaceHub` con
`tiiuae/falcon-7b-instruct`, que solo requiere un token gratuito.

**Nota sobre los embeddings**: el `worker.py` original usa
`HuggingFaceInstructEmbeddings` (una clase de LangChain hoy en desuso).
Aquí usamos su equivalente moderno, `HuggingFaceEmbeddings`, con el mismo
modelo (`sentence-transformers/all-MiniLM-L6-v2`) y los mismos argumentos —
el hueco a completar es idéntico en espíritu al original.

In [ ]:
# Instalamos las librerías necesarias.
!pip install -q langchain langchain-community chromadb sentence-transformers pypdf reportlab huggingface_hub

In [ ]:
# Generamos el mismo PDF de ejemplo que en el notebook de clase, para que
# esta práctica sea independiente y no necesites volver a ejecutar aquel.
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas

def crear_pdf_ejemplo(ruta="documento_ejemplo.pdf"):
    contenido = """Manual interno de Acme Corp - Politica de vacaciones y viajes

    Articulo 1. Dias de vacaciones.
    Todos los empleados a tiempo completo de Acme Corp tienen derecho a 22
    dias laborables de vacaciones pagadas por cada año completo trabajado.

    Articulo 2. Solicitud de vacaciones.
    Las vacaciones deben solicitarse con al menos 15 dias de antelacion a
    traves del portal interno de Recursos Humanos.

    Articulo 3. Reembolso de gastos de viaje.
    Los empleados que viajen por motivos de trabajo pueden solicitar el
    reembolso de transporte, alojamiento y comidas dentro de los 30 dias
    siguientes al viaje. El limite diario para comidas es de 40 dolares.
    """
    c = canvas.Canvas(ruta, pagesize=letter)
    texto = c.beginText(40, 750)
    texto.setFont("Helvetica", 11)
    for linea in contenido.strip().split("\n"):
        texto.textLine(linea.strip())
    c.drawText(texto)
    c.save()
    return ruta

document_path = crear_pdf_ejemplo()
print("PDF de ejemplo listo en:", document_path)

In [ ]:
import os
from getpass import getpass

if "HUGGINGFACEHUB_API_TOKEN" not in os.environ:
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = getpass(
        "Pega tu token gratuito de Hugging Face (https://huggingface.co/settings/tokens): "
    )

## Ejercicio: completa los 4 huecos de `worker.py`

Abajo tienes la misma estructura que el `worker.py` real: una función
`init_llm()` que crea el LLM y los embeddings, una función
`process_document(document_path)` que indexa el PDF y arma la cadena de RAG,
y una función `process_prompt(prompt)` que responde preguntas y guarda el
historial.

Completa cada `# TODO` sin mirar todavía la solución de más abajo.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.llms import HuggingFaceHub
from langchain.chains import RetrievalQA

DEVICE = "cpu"
embeddings_model_name = "sentence-transformers/all-MiniLM-L6-v2"

llm_hub = None
embeddings = None
conversation_retrieval_chain = None
chat_history = []


def init_llm():
    """Crea el LLM y el modelo de embeddings, una sola vez."""
    global llm_hub, embeddings

    llm_hub = HuggingFaceHub(
        repo_id="tiiuae/falcon-7b-instruct",
        model_kwargs={"temperature": 0.5, "max_new_tokens": 200},
    )

    # TODO: crea el modelo de embeddings con HuggingFaceEmbeddings, usando
    # embeddings_model_name como modelo y model_kwargs={"device": DEVICE}
    embeddings = None  # <-- reemplaza esto


def process_document(document_path):
    """Carga un PDF, lo trocea, lo indexa en Chroma y arma la cadena de RAG."""
    global conversation_retrieval_chain

    # TODO: carga el PDF con PyPDFLoader(document_path)
    loader = None  # <-- reemplaza esto
    documents = loader.load()

    # TODO: trocea el texto con RecursiveCharacterTextSplitter,
    # usando chunk_size=1024 y chunk_overlap=64
    text_splitter = None  # <-- reemplaza esto
    texts = text_splitter.split_documents(documents)

    db = Chroma.from_documents(texts, embeddings)
    retriever = db.as_retriever(search_type="mmr", search_kwargs={"k": 6, "lambda_mult": 0.25})

    conversation_retrieval_chain = RetrievalQA.from_chain_type(
        llm=llm_hub, chain_type="stuff", retriever=retriever, input_key="question"
    )


def process_prompt(prompt):
    """Hace una pregunta a la cadena de RAG y guarda el par pregunta-respuesta."""
    global chat_history

    output = conversation_retrieval_chain({"question": prompt, "chat_history": chat_history})
    answer = output["result"]

    # TODO: añade el par (prompt, answer) a chat_history
    # (pista: chat_history es una lista; añade una tupla con .append(...))

    return answer

## Probando tu solución

Si los 4 huecos están bien completados, esto debería inicializar el LLM,
indexar el PDF de ejemplo, y responder correctamente a una pregunta sobre su
contenido — y `chat_history` debería tener un elemento más después de cada
pregunta.

In [ ]:
init_llm()
process_document(document_path)

respuesta = process_prompt("¿Cuantos dias de vacaciones tengo por año?")
print("Respuesta:", respuesta)
print("Historial:", chat_history)

<details>
<summary>💡 Ver solución</summary>

Este es el código exacto de la solución real (`Worker_completed.py`,
adaptado a la variante gratuita con Hugging Face):

```python
def init_llm():
    global llm_hub, embeddings

    llm_hub = HuggingFaceHub(
        repo_id="tiiuae/falcon-7b-instruct",
        model_kwargs={"temperature": 0.5, "max_new_tokens": 200},
    )

    # Modelo de embeddings: convierte cada fragmento de texto en un vector.
    embeddings = HuggingFaceEmbeddings(
        model_name=embeddings_model_name,
        model_kwargs={"device": DEVICE},
    )


def process_document(document_path):
    global conversation_retrieval_chain

    # Cargamos el PDF y extraemos su texto.
    loader = PyPDFLoader(document_path)
    documents = loader.load()

    # Troceamos el texto en fragmentos de 1024 caracteres, con 64 de solape
    # entre fragmentos consecutivos (para no cortar una idea por la mitad).
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1024, chunk_overlap=64)
    texts = text_splitter.split_documents(documents)

    db = Chroma.from_documents(texts, embeddings)
    retriever = db.as_retriever(search_type="mmr", search_kwargs={"k": 6, "lambda_mult": 0.25})

    conversation_retrieval_chain = RetrievalQA.from_chain_type(
        llm=llm_hub, chain_type="stuff", retriever=retriever, input_key="question"
    )


def process_prompt(prompt):
    global chat_history

    output = conversation_retrieval_chain({"question": prompt, "chat_history": chat_history})
    answer = output["result"]

    # Guardamos el par pregunta-respuesta para mantener el historial de la
    # conversación, igual que hace la app real antes de devolver la respuesta.
    chat_history.append((prompt, answer))

    return answer
```
</details>

## Para seguir

- Notebook de clase: `notebooks_clase/clase10_rag_documentos.ipynb`
- Capítulo del manual: `manual/14_rag_documentos.md`

Reto extra (opcional): cambia `chunk_size=1024, chunk_overlap=64` por
`chunk_size=300, chunk_overlap=30` en `process_document`, vuelve a indexar el
PDF y compara las respuestas. ¿Notas alguna diferencia en la precisión o en
el nivel de detalle de las respuestas?